In [1]:
import sys
sys.path.append('..')

import os
os.environ['LOGURU_LEVEL'] = 'ERROR'

import torch.nn as nn
import torch
from transformer_maskgit import CTViT
from src.data import load_ct_image, ct_image2tensor

/home/borntowarn/projects/chest-diseases/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/borntowarn/projects/chest-diseases/venv/lib/python3.11/site-packages/vector_quantize_pytorch/vector_quantize_pytorch.py:261: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/home/borntowarn/projects/chest-diseases/venv/lib/python3.11/site-packages/vector_quantize_pytorch/vector_quantize_pytorch.py:391: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


In [2]:
class ProjectionVIT(nn.Module):
    def __init__(self):
        super(ProjectionVIT, self).__init__()
        self.VIT = CTViT(
            dim=512,
            codebook_size=8192,
            image_size=480,
            patch_size=20,
            temporal_patch_size=10,
            spatial_depth=4,
            temporal_depth=4,
            dim_head=32,
            heads=8,
        )
        self.projection_layer = nn.Linear(294912, 512, bias=False)

    def forward(self, x):
        x = self.VIT(x, return_encoded_tokens=True)

        x = torch.mean(x, dim=1)
        x = x.view(x.size(0), -1)

        x = self.projection_layer(x)
        return x

In [3]:
class MLP(nn.Module):
    def __init__(
            self,
            input_size=512,
            num_classes=18,
            activation='relu',
            hidden_sizes=[1024, 2048, 1024, 256, 128],
            dropout=0.1
        ):
        super().__init__()
        
        # Pick activation
        if activation == "relu":
            activation_cls = nn.ReLU
        elif activation == "leaky_relu":
            activation_cls = nn.LeakyReLU
        elif activation == "gelu":
            activation_cls = nn.GELU
        else:
            raise ValueError(f"Unsupported activation: {activation}")

        layers = []
        in_dim = input_size
        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.BatchNorm1d(h))  # helps stabilize
            layers.append(activation_cls())
            layers.append(nn.Dropout(dropout))
            in_dim = h

        # Final classification layer
        if len(hidden_sizes) == 0:
            layers.append(activation_cls())
            layers.append(nn.Dropout(dropout))
            
        layers.append(nn.Linear(in_dim, num_classes))

        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

In [4]:
model_lipro = ProjectionVIT()
model_lipro.load_state_dict(
    torch.load('/home/borntowarn/projects/chest-diseases/training/weights/CT-RATE/ProjectionVIT_LiPro_V2.pt')
)
model_lipro.to('cuda').eval()


ProjectionVIT(
  (VIT): CTViT(
    (spatial_rel_pos_bias): ContinuousPositionBias(
      (net): ModuleList(
        (0): Sequential(
          (0): Linear(in_features=2, out_features=512, bias=True)
          (1): LeakyReLU(negative_slope=0.1)
        )
        (1): Sequential(
          (0): Linear(in_features=512, out_features=512, bias=True)
          (1): LeakyReLU(negative_slope=0.1)
        )
        (2): Linear(in_features=512, out_features=8, bias=True)
      )
    )
    (to_patch_emb_first_frame): Sequential(
      (0): Rearrange('b c 1 (h p1) (w p2) -> b 1 h w (c p1 p2)', p1=20, p2=20)
      (1): LayerNorm((400,), eps=1e-05, elementwise_affine=True)
      (2): Linear(in_features=400, out_features=512, bias=True)
      (3): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (to_patch_emb): Sequential(
      (0): Rearrange('b c (t pt) (h p1) (w p2) -> b t h w (c pt p1 p2)', p1=20, p2=20, pt=10)
      (1): LayerNorm((4000,), eps=1e-05, elementwise_affine=True)
     

In [5]:
mlp = MLP(
    512,
    2,
    'gelu',
    [256,128],
    0.2
)
mlp.load_state_dict(torch.load('/home/borntowarn/projects/chest-diseases/training/weights/CT-RATE/model_binary.pth'))
mlp.to('cuda').eval()



MLP(
  (layers): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [6]:
# init dataset and dataloader
from torch.utils.data import Dataset

class CTRateTTADataset(Dataset):
    def __init__(self, pathes):
        super().__init__()

        self.pathes = pathes

    def __len__(self):
        return len(self.pathes)

    def __getitem__(self, idx):
        # change split
        path = self.pathes[idx]

        image = load_ct_image(path)

        return image, path.name

def custom_collate_fn(batch):
    """
    Custom function to collate data in DataLoader.
    batch: list of tuples (image, name)
    Returns:
        images (list of tio.ScalarImage): list of CT image objects
        names (list of str): image names
    """
    images, names = zip(*batch)
    return list(images), list(names)


In [7]:
from pathlib import Path
pathes = list(Path('/home/borntowarn/projects/chest-diseases/training/data/CT-RATE/dataset/valid_fixed').rglob('*.gz'))
dataset = CTRateTTADataset(pathes)

In [8]:
from unittest import result
import torch
import torchio as tio
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd
import json


def run_tta_inference(
    dataset,
    projection_model,
    mlp_model,
    device="cuda",
    num_workers=4,
    batch_size=1,
    tta_mode="mean",
    save_path="tta_predictions.json",
):
    """
    Runs TTA-based inference on a dataset of CT studies using ProjectionVIT + MLP.

    Saves only pathology probabilities (last neuron output of MLP).

    Args:
        dataset (Dataset): CTRateTTADataset
        projection_model (nn.Module): pretrained ProjectionVIT
        mlp_model (nn.Module): pretrained MLP
        device (str): device for inference
        num_workers (int): dataloader workers
        batch_size (int): batch size (usually 1 for 3D)
        tta_mode (str): 'mean' or 'vote'
        save_path (str): CSV output path

    Returns:
        pd.DataFrame: predictions with pathology probabilities
    """

    # Define TTA augmentations
    tta_transforms = [
        # tio.RandomAffine(scales=(0.9, 1.1), degrees=(0, 5), translation=(0, 5), p=1),
        # tio.RandomElasticDeformation(num_control_points=7, max_displacement=3, p=1),
        tio.RandomNoise(mean=0, std=(0, 0.05), p=1),
        tio.RandomGamma(log_gamma=(-0.4, 0.4), p=1),
        tio.RandomBlur(std=(0, 1.0), p=1),
        tio.RandomMotion(degrees=5, translation=2, num_transforms=2, p=1),
        tio.RandomBiasField(coefficients=0.3, p=1),
        # tio.RandomAnisotropy(axes=(0,1,2), downsampling=(1.0, 2.0), p=1),
        tio.RandomGhosting(num_ghosts=(1, 3), intensity=(0.1, 0.3), p=1),
        tio.RandomSpike(num_spikes=(1, 3), intensity=(0.1, 0.3), p=1),
        tio.RandomSwap(patch_size=15, num_iterations=20, p=1),
        tio.Lambda(lambda x: x)  # no change (original)
    ]

    loader = DataLoader(dataset, batch_size=batch_size, num_workers=num_workers, shuffle=False, collate_fn=custom_collate_fn)
    # loader = dataset

    projection_model.eval().to(device)
    mlp_model.eval().to(device)

    results = {}

    with torch.no_grad():
        for image, name in tqdm(loader, desc="TTA inference"):
            pathology_preds = []

            image = image[0]
            name = name[0]

            for aug in tta_transforms:
                # image = tio.ScalarImage(image.data.squeeze(0), image.affine)

                aug_img = aug(image)

                # Convert to tensor
                tensor = ct_image2tensor(aug_img).to(device).float()  # (1, D, H, W)
                # Forward
                features = projection_model(tensor.unsqueeze(0))
                probs = torch.softmax(mlp_model(features), dim=-1)

                # Take only last probability (pathology)
                pathology_preds.append(probs[0, -1].cpu().item())

            results[name] = pathology_preds
            json.dump(results, open(save_path, 'w'))


In [9]:
df = run_tta_inference(
    dataset,
    model_lipro,
    mlp,
    'cuda',
)

TTA inference:   0%|          | 8/3039 [18:58<119:50:06, 142.33s/it]


KeyboardInterrupt: 